# Data Preprocessing

> **Chapter 1.1 — Data Exploration**

Raw data is almost never analysis-ready. Our `flawed_dataset.csv` describes 1,050 employees and was deliberately corrupted with every common data-quality problem: missing values, duplicate rows, inconsistent text, impossible numbers, and invalid dates and emails.

In this notebook we **clean it step by step**, using the simplest code that does the job. By the end every column will be consistent, valid, and free of missing values.


---

## 1. Load and Inspect

Always start by *looking* at the data before changing anything.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("flawed_dataset.csv")
df.head()

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
# How many values are missing in each column?
df.isnull().sum()

---

## 2. Visualize the Missing Data

A heatmap shows *where* the gaps are at a glance — each yellow streak is a missing value.


In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing values (yellow = missing)")
plt.tight_layout()
plt.show()

---

## 3. Remove Duplicate Rows

Exact duplicates inflate counts and skew every statistic. Drop them first.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Rows after dropping duplicates:", len(df))

---

## 4. Standardize Inconsistent Text

The same category is written many ways. We lowercase/uppercase, strip spaces, and map every variant to one canonical value.


In [ ]:
# Gender: 'f', 'F', 'FEMALE', 'm', 'M', 'male'  →  'female' / 'male'
df["Gender"] = df["Gender"].str.lower().str.strip()
df["Gender"] = df["Gender"].replace({"f": "female", "m": "male"})
print("Gender:", df["Gender"].dropna().unique())

In [ ]:
# Department: 'hr', 'HR', 'ENG', 'Engineering', 'FIN', 'Finance'  →  full names
df["Department"] = df["Department"].str.upper().str.strip()
df["Department"] = df["Department"].replace({
    "HR": "Human Resources",
    "ENG": "Engineering", "ENGINEERING": "Engineering",
    "FIN": "Finance",     "FINANCE": "Finance",
})
print("Department:", df["Department"].dropna().unique())

In [ ]:
# Left_Company: 'Yes','Y','1','No','N','0'  →  numeric 1 / 0
df["Left_Company"] = df["Left_Company"].astype(str).str.lower().str.strip()
df["Left_Company"] = df["Left_Company"].map({
    "yes": 1, "y": 1, "1": 1,
    "no": 0,  "n": 0, "0": 0,
})
print("Left_Company:", df["Left_Company"].dropna().unique())

---

## 5. Replace Impossible Numbers with `NaN`

`Age` contains `-5` and `999`; `Salary` contains `-1000` and `1,000,000`. These are **sentinel** values — fake numbers standing in for missing data. We replace anything outside a sensible range with `NaN` so they can be imputed later.


In [ ]:
# Keep only plausible values; everything else becomes NaN
df.loc[~df["Age"].between(0, 100), "Age"] = np.nan
df.loc[~df["Salary"].between(10_000, 200_000), "Salary"] = np.nan

print("Age   range now:", df["Age"].min(), "to", df["Age"].max())
print("Salary range now:", df["Salary"].min(), "to", df["Salary"].max())

---

## 6. Parse Dates and Validate Emails

`pd.to_datetime` with `errors="coerce"` turns unparseable strings like `"not-a-date"` into `NaT`. A simple regex flags malformed emails.


In [ ]:
# Bad dates ('not-a-date') become NaT
df["Join_Date"] = pd.to_datetime(df["Join_Date"], errors="coerce")

# Invalid email formats become NaN
valid_email = df["Email"].fillna("").str.contains(r"^[\w.-]+@[\w.-]+\.\w+$")
df.loc[~valid_email, "Email"] = np.nan

print("Valid emails remaining:", df["Email"].notna().sum())

---

## 7. Tidy Up Names

Names appear as `BOB`, `eve`, `Alice` — fix the casing with `.str.title()`.


In [ ]:
df["Name"] = df["Name"].str.title()
print("Name examples:", df["Name"].dropna().unique()[:6])

---

## 8. Fill the Remaining Missing Values

The right fill depends on the column:

- **Numbers** → median (robust to outliers)
- **Salary** → median *within each department* (more accurate than a global median)
- **Categories** → an explicit `"Unknown"` label


In [ ]:
# Numeric columns → median
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Feedback_Score"] = df["Feedback_Score"].fillna(df["Feedback_Score"].median())

# Salary → department-wise median, then any leftovers → global median
df["Salary"] = df.groupby("Department")["Salary"].transform(lambda s: s.fillna(s.median()))
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

# Categorical columns → 'Unknown'
df["Name"] = df["Name"].fillna("Unknown")
df["Gender"] = df["Gender"].fillna("unknown")
df["Department"] = df["Department"].fillna("Unknown")

df.isnull().sum()

```{admonition} Email and Join_Date
:class: note
We deliberately leave `Email` and `Join_Date` with missing values — there is no sensible way to invent an email address or a hire date, so a genuine `NaN`/`NaT` is more honest than a fake fill.
```


---

## 9. Verify the Result

A final look confirms the data is now consistent and (for the columns we imputed) complete.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df["Age"], bins=20, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Age — cleaned")

sns.boxplot(x=df["Salary"], ax=axes[1], color="seagreen")
axes[1].set_title("Salary — cleaned")

sns.countplot(x="Gender", data=df, ax=axes[2], color="indianred")
axes[2].set_title("Gender — standardized")

plt.tight_layout()
plt.show()

In [ ]:
print("Final shape:", df.shape)
df.head()

---

## Summary

```{list-table}
:header-rows: 1
:widths: 30 70

* - Step
  - What it fixed
* - Drop duplicates
  - Removed exact repeat rows
* - Standardize text
  - One spelling per category (Gender, Department, Left_Company)
* - Replace sentinels
  - Turned impossible numbers (-5, 999, 1,000,000) into `NaN`
* - Parse dates / emails
  - Invalid entries became `NaT` / `NaN`
* - Fill missing
  - Median for numbers, `"Unknown"` for categories
```

```{admonition} What's Next
:class: note
The cleaned data still has features on very different scales. Chapter 1.4 — **Data Transformation** — rescales them so models can use every feature fairly.
```
